In [ ]:
# Install/upgrade dependencies
!pip install -q gradio tensorflow

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pathlib
import PIL
import tensorflow as tf
import gradio as gr

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 1. Load Dataset

In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

data_dir = tf.keras.utils.get_file(
    'flower_photos',
    origin=dataset_url,
    untar=True
)

# The downloaded folder already contains 'flower_photos' subfolder
data_dir = pathlib.Path(data_dir)
if not (data_dir / 'roses').exists():
    data_dir = data_dir / 'flower_photos'

# Count images per class
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
print("Classes and image counts:")
for d in class_dirs:
    print(f"  {d.name}: {len(list(d.glob('*')))} images")

total = sum(len(list(d.glob('*'))) for d in class_dirs)
print(f"\nTotal images: {total}")

## 2. Create Train / Validation Datasets

In [ ]:
IMG_HEIGHT = 224   # MobileNetV2 native size (better than 180)
IMG_WIDTH  = 224
BATCH_SIZE = 32
SEED       = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Class names:", class_names)

## 3. Compute Class Weights (fixes bias toward majority class)

In [ ]:
# Count samples per class in training split
class_counts = {}
for d in class_dirs:
    name = d.name
    if name in class_names:
        class_counts[name] = len(list(d.glob('*')))

total_train = sum(class_counts.values()) * 0.8   # approx 80% for training
class_weight = {}
for idx, name in enumerate(class_names):
    count = class_counts.get(name, 1)
    # weight = total / (num_classes * class_count)
    class_weight[idx] = total_train / (NUM_CLASSES * count * 0.8)

print("Class weights:")
for idx, name in enumerate(class_names):
    print(f"  {name}: {class_weight[idx]:.3f}")

## 4. Data Augmentation & Pipeline

In [ ]:
# Augmentation (applied only during training)
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE

# ✅ DO NOT normalize here — MobileNetV2 has its own preprocess_input
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Visualize some augmented samples
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    aug_images = data_augmentation(images[:9], training=True)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(aug_images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=10)
        plt.axis("off")
plt.suptitle("Augmented Training Samples", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build Model with MobileNetV2 Transfer Learning

**Key fix:** MobileNetV2's `preprocess_input` expects pixel values in `[0, 255]` (it internally scales to `[-1, 1]`).  
Using `Rescaling(1./255)` before it was causing **double normalization** → garbage predictions → model collapsed to always predicting the majority class (dandelion).

In [ ]:
# Load MobileNetV2 base (frozen)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # freeze for initial training

inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))

# Augmentation (only active during training)
x = data_augmentation(inputs)

# ✅ CORRECT: Use MobileNetV2's own preprocessing (scales [0,255] → [-1,1])
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

# Base model
x = base_model(x, training=False)   # training=False keeps BN in inference mode

# Classification head
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

## 6. Phase 1 — Train Classification Head Only

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Phase 1: Training classification head...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop, reduce_lr]
)

## 7. Phase 2 — Fine-tune Top Layers of MobileNetV2

In [ ]:
# Unfreeze the top 30 layers of the base model
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Fine-tuning from layer {fine_tune_at} / {len(base_model.layers)}")
print(f"Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}")

# Recompile with a MUCH lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop2 = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("\nPhase 2: Fine-tuning...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=[early_stop2]
)

## 8. Plot Training History

In [ ]:
def plot_history(h1, h2=None):
    acc     = h1.history['accuracy']
    val_acc = h1.history['val_accuracy']
    loss    = h1.history['loss']
    val_loss= h1.history['val_loss']

    if h2:
        acc      += h2.history['accuracy']
        val_acc  += h2.history['val_accuracy']
        loss     += h2.history['loss']
        val_loss += h2.history['val_loss']

    epochs_range = range(len(acc))
    split = len(h1.history['accuracy'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, acc,     label='Train Accuracy')
    axes[0].plot(epochs_range, val_acc, label='Val Accuracy')
    if h2: axes[0].axvline(split - 1, color='gray', linestyle='--', label='Fine-tune start')
    axes[0].set_title('Accuracy'); axes[0].legend()

    axes[1].plot(epochs_range, loss,     label='Train Loss')
    axes[1].plot(epochs_range, val_loss, label='Val Loss')
    if h2: axes[1].axvline(split - 1, color='gray', linestyle='--', label='Fine-tune start')
    axes[1].set_title('Loss'); axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

## 9. Evaluate on Validation Set

In [ ]:
loss, acc = model.evaluate(val_ds, verbose=1)
print(f"\n✅ Final Validation Accuracy: {acc*100:.2f}%")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

## 10. Save the Model

In [ ]:
model.save('flower_classifier.keras')
print("✅ Model saved as flower_classifier.keras")

# Save class names for later use
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print("✅ Class names saved as class_names.json")

## 11. Gradio Demo

In [ ]:
import gradio as gr
import numpy as np
import tensorflow as tf

# Load saved model & class names (run this cell independently if needed)
# model = tf.keras.models.load_model('flower_classifier.keras')
# import json
# with open('class_names.json') as f:
#     class_names = json.load(f)

FLOWER_EMOJIS = {
    'daisy':      '🌼',
    'dandelion':  '🌻',
    'roses':      '🌹',
    'sunflowers': '🌸',
    'tulips':     '🌷',
}

def predict_image(img):
    """
    img: numpy array of shape (H, W, 3), pixel values in [0, 255]
    """
    # Resize to model input size
    img_tensor = tf.image.resize(img, (IMG_HEIGHT, IMG_WIDTH))

    # ✅ DO NOT divide by 255 — preprocess_input handles normalization internally
    img_tensor = tf.expand_dims(img_tensor, axis=0)   # add batch dim → (1, 224, 224, 3)

    # Predict
    predictions = model.predict(img_tensor, verbose=0)
    scores = predictions[0]   # softmax already applied in model

    # Return label → confidence dict with emoji
    return {
        f"{FLOWER_EMOJIS.get(class_names[i], '🌺')} {class_names[i]}": float(scores[i])
        for i in range(NUM_CLASSES)
    }

# Build Gradio interface
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy", label="Upload a flower image"),
    outputs=gr.Label(num_top_classes=NUM_CLASSES, label="Predictions"),
    title="🌸 Flower Classifier",
    description="Upload an image of a flower and the model will classify it into: Daisy, Dandelion, Rose, Sunflower, or Tulip.",
    examples=None,
    flagging_mode="never"
)

demo.launch(debug=True, share=True)